In [1]:
import os
from circuit_tracer.subgraph.prune import prune_graph_pipeline, PruneGraph
# from circuit_tracer.subgraph.group_llm import grouping_pipeline
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import shap

## Load LLM

In [2]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Use SHAP for token weights

In [4]:
explainer = shap.Explainer(model, tokenizer)

In [5]:
prompts = ['Fact: The capital of the state containing Dallas is']

In [ ]:
shap_values = explainer(prompts)

In [ ]:
shap.plots.text(shap_values)

In [52]:
print(shap_values)

.values =
array([[[-4.22446732e-06],
        [ 6.54286429e+00],
        [ 1.68508452e+00],
        [ 1.20230146e+00],
        [ 6.04736818e+00],
        [ 6.77137808e-02],
        [ 4.40673760e+00],
        [ 4.12085123e-01],
        [ 4.40843961e-01]]])

.base_values =
array([[-21.73993724]])

.data =
(array(['', 'Cat', ' is', ' to', ' kitten', ' as', ' dog', ' is', ' to'],
      dtype=object),)


In [34]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze()) # How to normalize Shap values? softmax, relu first then normalize, ...
print(token_weights)

[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
 0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]


## Generate graph if not exist

In [4]:
from generate_new_graphs import download_graph
download_graph(
    prompt=prompts[0],
    slug="austin_plt",
    source_set = 'gemmascope-transcoder-16k',
    out_path="temp_graph_files/austin_plt.json",
    node_threshold = 0.95,
    edge_threshold = 0.98,
)

requesting graph for slug='austin_plt' prompt='Fact: The capital of the state containing Dallas is...'


downloading graph json from https://neuronpedia-attrib.s3.us-east-1.amazonaws.com/user-graphs/anonymous/austin_plt.json
saved austin_plt -> temp_graph_files/austin_plt.json (nodes=5387)


## ASG Pipeline

### Prune

In [3]:
import circuit_tracer, circuit_tracer.subgraph.prune as p, circuit_tracer.graph as g; print(circuit_tracer.__file__); print(p.__file__); print(g.__file__)

d:\anaconda\envs\circuit\Lib\site-packages\circuit_tracer\__init__.py
d:\anaconda\envs\circuit\Lib\site-packages\circuit_tracer\subgraph\prune.py
d:\anaconda\envs\circuit\Lib\site-packages\circuit_tracer\graph.py


In [ ]:
from circuit_tracer.subgraph.prune import prune_graph_pipeline

# prune_graph = load_prune_graph('subgraph/puppy_clt_shap_07_085.pt')
token_weights = [0,0,0,0,1/3,0,0,1/3,0,1/3,0]
# 1) Build prune_graph as usual
prune_graph = prune_graph_pipeline(
    json_path="temp_graph_files/austin_plt.json",
    token_weights = token_weights,
    logit_weights="target",
    combined_scores_method="geometric",
    node_threshold=0.5,
    edge_threshold=0.95,
    alpha=0.55,
    keep_all_tokens_and_logits=False,
    filter_act_density=True,
)


In [3]:
prune_graph.num_nodes, prune_graph.num_edges

(55, 690)

In [4]:
for node_id in prune_graph.kept_ids:
    print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_scores[prune_graph.kept_ids.index(node_id)].item())


4_6931_9 website references 0.007538920734077692
4_13154_9 Texas 0.010859806090593338
7_6861_9 Texas legal cases 0.008748438209295273
14_2268_9 Texas 0.014798968099057674
16_25_9 Texas legal documents 0.018482232466340065
16_12678_9 cities 0.005874766036868095
16_4298_10 capital 0.0063041141256690025
17_7178_10 government buildings 0.0069392467848956585
18_1437_10 Legal documents from Texas 0.011999973095953465
18_5495_10 locations 0.00968110654503107
18_6101_10 capital cities 0.014354558661580086
18_8959_10 government/state 0.01550628524273634
19_7477_9 Dallas 0.009892446920275688
19_37_10 Places 0.006881047505885363
19_1445_10 Downtowns of cities 0.011983275413513184
19_2695_10 cities 0.015295961871743202
19_7477_10 Dallas 0.01141655258834362
20_15589_9 Texas 0.027857357636094093
20_114_10 Oklahoma locations 0.006359212566167116
20_5916_10 capital 0.007362022064626217
20_7507_10 countries 0.006602002307772636
20_15589_10 Texas 0.05035414546728134
21_5943_10 cities 0.03369581326842308

In [4]:
print(prune_graph.node_scores.max(), prune_graph.node_scores.min())
print(torch.sum(prune_graph.node_scores))

tensor(0.5892) tensor(0.0104)


NameError: name 'torch' is not defined

In [1]:
from circuit_tracer.subgraph.prune import save_prune_graph, load_prune_graph
# save_prune_graph(prune_graph, "subgraph/austin_plt_clean_5_95_55.pt")
prune_graph = load_prune_graph("subgraph/austin_plt_clean_5_95_55.pt")

In [17]:
from circuit_tracer.subgraph.cluster import cluster_graph, cluster_graph_with_labels
from circuit_tracer.subgraph.auto_grouping import find_best_k
# 2) Auto-select k
best_k, sweep = find_best_k(prune_graph, max_layer_span=4, gamma=10, mediation_penalty=0.1, similarity=None)

print("best_k:", best_k)
print("best score:", sweep[best_k]["total"] if best_k in sweep else None)

# 3) Run final clustering with best_k
supernodes = cluster_graph_with_labels(
    prune_graph,
    target_k=12,
    max_layer_span=4,
)

best_k: 2
best score: 0.7878301473453532


In [18]:
supernodes

[['cluster_1', '4_13154_9', '7_6861_9'],
 ['cluster_3', '16_4298_10', '17_7178_10'],
 ['cluster_4', '16_12678_9', '18_1437_10', '16_25_9'],
 ['cluster_5', '18_5495_10', '19_1445_10', '19_37_10', '19_2695_10'],
 ['cluster_6', '18_6101_10', '18_8959_10'],
 ['cluster_7', '19_7477_9', '19_7477_10'],
 ['cluster_9', '20_15589_9', '20_15589_10', '20_114_10'],
 ['cluster_10', '21_14975_10', '20_5916_10'],
 ['cluster_12', '22_3064_10', '21_6795_10'],
 ['cluster_13', '22_3551_10', '22_11718_10'],
 ['cluster_16', '23_6617_10', '24_6394_10'],
 ['cluster_17', '23_13193_10', '23_15366_10', '23_12918_10', '23_2288_10'],
 ['cluster_18', '23_13841_10', '23_11444_10', '23_12237_10'],
 ['cluster_20', '25_4679_10', '25_13300_10', '24_6044_10'],
 ['cluster_21',
  '24_709_10',
  '25_4259_10',
  '25_583_10',
  '25_762_10',
  '25_2687_10',
  '24_15694_10',
  '24_15627_10',
  '24_15013_10']]

In [19]:
for supernode in supernodes:
    print(supernode[0])
    for node_id in supernode[1:]:
        print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_scores[prune_graph.kept_ids.index(node_id)].item())

cluster_1
4_13154_9 Texas 0.010859806090593338
7_6861_9 Texas legal cases 0.008748438209295273
cluster_3
16_4298_10 capital 0.0063041141256690025
17_7178_10 government buildings 0.0069392467848956585
cluster_4
16_12678_9 cities 0.005874766036868095
18_1437_10 Legal documents from Texas 0.011999973095953465
16_25_9 Texas legal documents 0.018482232466340065
cluster_5
18_5495_10 locations 0.00968110654503107
19_1445_10 Downtowns of cities 0.011983275413513184
19_37_10 Places 0.006881047505885363
19_2695_10 cities 0.015295961871743202
cluster_6
18_6101_10 capital cities 0.014354558661580086
18_8959_10 government/state 0.01550628524273634
cluster_7
19_7477_9 Dallas 0.009892446920275688
19_7477_10 Dallas 0.01141655258834362
cluster_9
20_15589_9 Texas 0.027857357636094093
20_15589_10 Texas 0.05035414546728134
20_114_10 Oklahoma locations 0.006359212566167116
cluster_10
21_14975_10 state/states 0.007017434574663639
20_5916_10 capital 0.007362022064626217
cluster_12
22_3064_10 Texas/Dallas 0.0

In [7]:
import json
from typing import List
def extract_supernodes(flow_map_path: str) -> List[List[str]]:
    """
    Load a mapping JSON like flow_analysis_supernode_map.json and return
    a list of supernodes in the format:
      [ ["SN_LABEL", "node_a", "node_b", ...], ... ]
    """
    p = Path(flow_map_path)
    with p.open("r", encoding="utf-8") as f:
        mapping = json.load(f)

    supernodes = [[label, *nodes] for label, nodes in mapping.items()]
    return supernodes

### Visualize on the web

In [20]:
import json
from circuit_tracer.subgraph.api import save_subgraph

status, body = save_subgraph(
    modelId="gemma-2-2b",
    slug="austin-plt-full",                       # parent graph slug
    displayName="activation-weight-group",     # subgraph name shown on Neuronpedia
    pinnedIds=prune_graph.kept_ids,                  # or PruneGraph.kept_ids
    supernodes=supernodes,               # output from grouping pipeline
    pruningThreshold=0.8,
    densityThreshold=0.99,
)

print("status:", status)
try:
    print(json.loads(body))
except Exception:
    print(body)

{'Content-Type': 'application/json', 'x-api-key': 'sk-np-Vi0OcQl8zNm9jC7nyGRYfwssvSakMyrPjV675uEWIuU0'}
status: 200
{'success': True, 'subgraphId': 'cmo2sl10o000012ga80lqreyf'}


# TODO
List of things that need more experiments/improvements in descending order of importance for each step
## Pruning
1. Choose threshold (currently just sweep the threshold until we have ~30 nodes)
2. Initialize with Shap (normalize) (softmax | relu + normalize | +base_value then normalize)
3. Normalize adjacency matrix (currently adjacency.abs() then normalize (negative edges still contribute to influence and relevance))

## Grouping

Objectives to group features:
- similar embeddings (in classic SAE usually decoder vectors)
- simliar edges/logit effects (promoting the same feature or the same logit)
- similar contexts (activating on similar tokens, concepts, contexts)
- based on the claim we make about the mechanism (we might want to preserve the paths)

Approaches tried:
- Greedy constraint grouping
    + built distance graph based on BERT embeddings of autointerp (+ feature type) and group with antichain constraint (don't group nodes with path to each other)
    + input: autointerp + adj matrix
    + problems: constraints too strict, sensitive on the edges of the graph; did not consider local role of the features.
- Prompt LLM: 
    + Classify feature type based on feature details
    + Group using feature types + autointerp
    + problems: cover context and global role of the features, but not local role (in this exact graph) -> might not be mechanisticly sensible
    
Approaches to try:
- Clustering by (decoder vectors or autointerp) + edge weights similarity, constraint by difference in layers (may be sweep diff_layer = 0, 1, ... and merge iteratively)
- Higher order graphs




## Eval
1. Test intervention api
2. Find dataset
3. Hypothesis testing: Mechanism Preservation, Mechanism Localization, Minimality